In [24]:
import pandas as pd
import numpy as np
# Carga del dataset transaccional
# Cargar el dataset crudo
df_productos = pd.read_csv('Raw/productos_raw.csv')

# Mostrar las primeras 5 filas para detectar a simple vista la "suciedad" de los datos
display(df_productos.head())

,id_producto,nombre_producto,marca,categoria,costo_unitario,precio_lista,stock_actual,activo,margen_ganancia,proveedor,cod_interno,ubicacion_deposito,unidad_medida,fecha_ingreso,peso_kg
0,101,Televisor 42'',Sony,hogar y deco,$2676.44,NaN,87,NO,0.4379,prov_a,INT-7407,DEP1,U,26-02-2024,3.95 kg
1,102,Auriculares BT,Adidas,Deporte,$2420.13,3052.45,34,no,0.2915,PROV B,INT-7221,DEP2,u,2023-05-12,8.42 kg
2,103,Zapatillas Running,LG,Indumentaria,$1360.93,2778.60,75,NO,24.6%,Prov. A,INT-6337,Deposito 1,unidad,2024-01-07,13.33 kg
3,104,Remera Deportiva,Philips,HOGAR,$1821.59,3570.69,48,No,-0.05,PROVEEDOR A,INT-5761,EXTERNO,UNIDAD,16/08/2023,23.69
4,105,Licuadora 600W,Sony,Hogar,$7567.49,9217.83,77,no,0.3358,Proveedor B,INT-6786,Deposito 1,u,2023-08-24,6.22 kg


In [25]:
# Vemos los tipos de datos iniciales (seguramente casi todo sea "object" por la falta de limpieza)
print("--- Información Estructural Inicial ---")
df_productos.info()

--- Información Estructural Inicial ---
<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_producto         20 non-null     int64  
 1   nombre_producto     20 non-null     str    
 2   marca               20 non-null     str    
 3   categoria           20 non-null     str    
 4   costo_unitario      20 non-null     str    
 5   precio_lista        15 non-null     float64
 6   stock_actual        20 non-null     int64  
 7   activo              20 non-null     str    
 8   margen_ganancia     19 non-null     str    
 9   proveedor           20 non-null     str    
 10  cod_interno         20 non-null     str    
 11  ubicacion_deposito  20 non-null     str    
 12  unidad_medida       20 non-null     str    
 13  fecha_ingreso       20 non-null     str    
 14  peso_kg             20 non-null     str    
dtypes: float64(1), int64(2), str(1

In [26]:
# 1. Identificadores
df_productos['id_producto'] = df_productos['id_producto'].astype(str)
df_productos['cod_interno'] = df_productos['cod_interno'].astype(str)

# 2. Limpieza de columnas monetarias y de medida (quitar símbolos y convertir a float)
df_productos['costo_unitario'] = df_productos['costo_unitario'].astype(str).str.replace('$', '', regex=False).str.replace(',', '').astype(float)
df_productos['peso_kg'] = df_productos['peso_kg'].astype(str).str.replace(' kg', '', case=False, regex=False).astype(float)

# 3. Limpieza del margen de ganancia (unificar formatos % y decimales a float)
def limpiar_margen(val):
    val_str = str(val).strip()
    if val_str.lower() == 'nan': return np.nan
    if '%' in val_str:
        return float(val_str.replace('%', '')) / 100
    try:
        return float(val)
    except:
        return np.nan

df_productos['margen_ganancia'] = df_productos['margen_ganancia'].apply(limpiar_margen)

# 4. Normalización de variables categóricas (texto a minúsculas y tipo 'category')
cols_categoricas = ['marca', 'categoria', 'proveedor', 'ubicacion_deposito', 'unidad_medida']
for col in cols_categoricas:
    df_productos[col] = df_productos[col].astype(str).str.lower().str.strip()
    # Para los nulos que se convirtieron a "nan" string, los volvemos a nulos reales
    df_productos[col] = df_productos[col].replace('nan', np.nan)
    df_productos[col] = df_productos[col].astype('category')

# 5. Estandarización de variable booleana (activo)
df_productos['activo'] = df_productos['activo'].astype(str).str.lower().str.strip()
df_productos['activo'] = df_productos['activo'].replace({'si': True, '1': True, 'no': False, '0': False, 'nan': pd.NA})


print("--- Tipos de Datos Post-Limpieza ---")
display(df_productos.dtypes)

--- Tipos de Datos Post-Limpieza ---


id_producto                str
nombre_producto            str
marca                 category
categoria             category
costo_unitario         float64
precio_lista           float64
stock_actual             int64
activo                  object
margen_ganancia        float64
proveedor             category
cod_interno                str
ubicacion_deposito    category
unidad_medida         category
fecha_ingreso              str
peso_kg                float64
dtype: object

In [27]:
# Chequeo de valores nulos 
print("--- Conteo de Valores Nulos ---")
display(df_productos.isnull().sum())

# Chequeo de duplicados
duplicados_totales = df_productos.duplicated().sum()
ids_duplicados = df_productos.duplicated(subset=['id_producto']).sum()

print("\n--- Análisis de Duplicados ---")
print(f"Filas exactamente duplicadas: {duplicados_totales}")
print(f"IDs de producto repetidos: {ids_duplicados}")

--- Conteo de Valores Nulos ---


id_producto           0
nombre_producto       0
marca                 0
categoria             0
costo_unitario        0
precio_lista          5
stock_actual          0
activo                0
margen_ganancia       1
proveedor             0
cod_interno           0
ubicacion_deposito    0
unidad_medida         0
fecha_ingreso         0
peso_kg               0
dtype: int64


--- Análisis de Duplicados ---
Filas exactamente duplicadas: 0
IDs de producto repetidos: 0


In [28]:
# Separamos el resumen para métricas numéricas y categóricas para mayor legibilidad
print("--- Resumen Estadístico (Numéricas) ---")
display(df_productos.describe().round(2))

print("\n--- Resumen Estadístico (Categóricas) ---")
display(df_productos.describe(include=['category']).T)

--- Resumen Estadístico (Numéricas) ---


,costo_unitario,precio_lista,stock_actual,margen_ganancia,peso_kg
count,20.00,15.00,20.00,19.00,20.00
mean,3801.96,5710.37,99.80,4.65,12.75
std,2071.34,2690.45,52.77,12.84,7.84
min,1360.93,2499.78,22.00,-0.05,2.24
25%,2409.54,3400.32,68.25,0.23,6.22
50%,3281.27,5136.16,96.00,0.29,12.16
75%,4584.61,8257.13,140.75,0.44,18.64
max,8898.79,10819.27,187.00,41.80,24.75



--- Resumen Estadístico (Categóricas) ---


,count,unique,top,freq
marca,20,10,adidas,3
categoria,20,8,electrónica,5
proveedor,20,7,proveedor a,7
ubicacion_deposito,20,6,deposito 1,8
unidad_medida,20,3,unidad,11


In [29]:
# 1. Normalización de Marca (Primera letra en mayúscula)
# Usamos .str.title() para que quede, por ejemplo, "Sony", "Adidas"
df_productos['marca'] = df_productos['marca'].astype(str).str.strip().str.title()

# 2. Unificación y normalización de Categorías
# Primero pasamos a minúsculas y sacamos espacios para facilitar el reemplazo
df_productos['categoria'] = df_productos['categoria'].astype(str).str.lower().str.strip()
# Reemplazamos los valores según tu regla
diccionario_categorias = {
    'deporte': 'deportes',
    'hogar y deco': 'hogar'
}
df_productos['categoria'] = df_productos['categoria'].replace(diccionario_categorias)
# Finalmente, ponemos la primera letra en mayúscula
df_productos['categoria'] = df_productos['categoria'].str.title()

# 3. Unificación de Proveedores
# Creamos una pequeña función para detectar la letra del proveedor independientemente de cómo esté escrito
def unificar_proveedor(prov):
    prov_limpio = str(prov).lower().strip()
    if 'a' in prov_limpio: return 'Proveedor A'
    if 'b' in prov_limpio: return 'Proveedor B'
    if 'c' in prov_limpio: return 'Proveedor C'
    return prov # Por si aparece uno nuevo no contemplado

df_productos['proveedor'] = df_productos['proveedor'].apply(unificar_proveedor)

# 4. Unificación de Depósitos
def unificar_deposito(dep):
    dep_limpio = str(dep).lower().strip()
    if '1' in dep_limpio: return 'Deposito 1'
    if '2' in dep_limpio: return 'Deposito 2'
    if 'ext' in dep_limpio: return 'Externo'
    return dep

df_productos['ubicacion_deposito'] = df_productos['ubicacion_deposito'].apply(unificar_deposito)

# 5. Estandarizar Unidad de Medida
# Directamente sobrescribimos toda la columna con el valor fijo
df_productos['unidad_medida'] = 'Unidad'

# 6. Revisión y tratamiento de Códigos Internos duplicados
# Primero contamos cuántos duplicados hay
duplicados_cod = df_productos.duplicated(subset=['cod_interno']).sum()
print(f"Se encontraron {duplicados_cod} códigos internos duplicados.")

# Si existen duplicados, nos quedamos con la primera aparición y eliminamos el resto
if duplicados_cod > 0:
    df_productos = df_productos.drop_duplicates(subset=['cod_interno'], keep='first')
    print("Los registros con códigos internos duplicados han sido eliminados.")

df_productos['fecha_ingreso'] = pd.to_datetime(
    df_productos['fecha_ingreso'],
    format='mixed',
    dayfirst=True
)


# Verificamos los resultados de nuestras transformaciones
print("\n--- Muestra de los datos limpios ---")
display(df_productos[['marca', 'categoria', 'proveedor', 'ubicacion_deposito', 'unidad_medida', 'fecha_ingreso']].head(10))

Se encontraron 0 códigos internos duplicados.

--- Muestra de los datos limpios ---


,marca,categoria,proveedor,ubicacion_deposito,unidad_medida,fecha_ingreso
0,Sony,Hogar,Proveedor A,Deposito 1,Unidad,2024-02-26
1,Adidas,Deportes,Proveedor B,Deposito 2,Unidad,2023-12-05
2,Lg,Indumentaria,Proveedor A,Deposito 1,Unidad,2024-07-01
3,Philips,Hogar,Proveedor A,Externo,Unidad,2023-08-16
4,Sony,Hogar,Proveedor B,Deposito 1,Unidad,2023-08-24
5,Reebok,Deportes,Proveedor A,Externo,Unidad,2023-12-30
6,Philips,Electrónica,Proveedor C,Deposito 1,Unidad,2023-01-18
7,Samsung,Deportes,Proveedor A,Deposito 2,Unidad,2023-08-24
8,Whirlpool,Deportes,Proveedor A,Deposito 1,Unidad,2022-01-24
9,Molinos,Indumentaria,Proveedor A,Deposito 1,Unidad,2024-02-14


In [30]:
# 1. Definimos un diccionario maestro (Hardcoding controlado) 
# Mapeamos el nombre exacto del producto hacia una tupla: (Marca Lógica, Categoría Lógica)
diccionario_maestro = {
    "Televisor 42''": ("Sony", "Electrónica"),
    "Auriculares BT": ("Sony", "Electrónica"),
    "Zapatillas Running": ("Nike", "Deportes"),
    "Remera Deportiva": ("Adidas", "Indumentaria"),
    "Licuadora 600W": ("Philips", "Hogar"),
    "Plancha de Pelo": ("Philips", "Hogar"),
    "Galletitas Integrales": ("Molinos", "Alimentos"),
    "Yerba 1kg": ("Molinos", "Alimentos"),
    "Short Entrenamiento": ("Reebok", "Indumentaria"),
    "Campera Impermeable": ("Topper", "Indumentaria"),
    "Parlante Portátil": ("LG", "Electrónica"),
    "Mouse Inalámbrico": ("Samsung", "Electrónica"),
    "Teclado Mecánico": ("Samsung", "Electrónica"),
    "Silla Ergonómica": ("Whirlpool", "Hogar"),
    "Mochila 25L": ("Nike", "Deportes"),
    "Botella Térmica": ("Topper", "Deportes"),
    "Camiseta Fútbol": ("Adidas", "Deportes"),
    "Zapatillas Urbanas": ("Reebok", "Indumentaria"),
    "Tablet 10''": ("Samsung", "Electrónica"),
    "Smart Watch": ("LG", "Electrónica")
}

# 2. Aplicamos el mapeo utilizando map() sobre la columna nombre_producto
# Usamos funciones lambda para extraer el primer elemento (marca) y el segundo (categoria)
df_productos['marca'] = df_productos['nombre_producto'].map(lambda x: diccionario_maestro[x][0] if x in diccionario_maestro else x)
df_productos['categoria'] = df_productos['nombre_producto'].map(lambda x: diccionario_maestro[x][1] if x in diccionario_maestro else x)

# 3. Volvemos a castear a tipo 'category' por buena práctica de memoria, ya que los sobrescribimos
df_productos['marca'] = df_productos['marca'].astype('category')
df_productos['categoria'] = df_productos['categoria'].astype('category')

# 4. Verificación visual rápida para confirmar que se arregló el cruce de datos
print("--- Catálogo Refactorizado ---")
display(df_productos[['nombre_producto', 'marca', 'categoria']])

--- Catálogo Refactorizado ---


,nombre_producto,marca,categoria
0,Televisor 42'',Sony,Electrónica
1,Auriculares BT,Sony,Electrónica
2,Zapatillas Running,Nike,Deportes
3,Remera Deportiva,Adidas,Indumentaria
4,Licuadora 600W,Philips,Hogar
5,Plancha de Pelo,Philips,Hogar
6,Galletitas Integrales,Molinos,Alimentos
7,Yerba 1kg,Molinos,Alimentos
8,Short Entrenamiento,Reebok,Indumentaria
9,Campera Impermeable,Topper,Indumentaria


In [31]:
# 1. Asegurarnos de que precio_lista sea numérico (por si se importó como texto)
df_productos['precio_lista'] = pd.to_numeric(df_productos['precio_lista'], errors='coerce')

# 2. Creamos una Serie temporal con el cálculo teórico del precio para TODOS los productos
# Fórmula: Precio = Costo * (1 + Margen)
precio_calculado = df_productos['costo_unitario'] * (1 + df_productos['margen_ganancia'])

# 3. Usamos fillna() para rellenar SOLAMENTE los valores nulos de precio_lista con nuestro cálculo
df_productos['precio_lista'] = df_productos['precio_lista'].fillna(precio_calculado)

# 4. Redondeamos a 2 decimales por prolijidad financiera
df_productos['precio_lista'] = df_productos['precio_lista'].round(2)

# 5. Verificamos cómo quedaron los 5 productos que mencionaste
# Filtramos por los nombres específicos para comprobar el resultado
productos_nulos = [
    "Televisor 42''", 
    "Galletitas Integrales", 
    "Short Entrenamiento", 
    "Campera Impermeable", 
    "Teclado Mecánico"
]

print("--- Precios Restaurados ---")
display(df_productos[df_productos['nombre_producto'].isin(productos_nulos)][['nombre_producto', 'costo_unitario', 'margen_ganancia', 'precio_lista']])

--- Precios Restaurados ---


,nombre_producto,costo_unitario,margen_ganancia,precio_lista
0,Televisor 42'',2676.44,0.4379,3848.45
6,Galletitas Integrales,3219.93,0.2135,3907.39
8,Short Entrenamiento,3400.90,0.2930,4397.36
9,Campera Impermeable,7546.56,41.8000,322992.77
12,Teclado Mecánico,4797.16,0.4486,6949.17


In [32]:
# 1. Calculamos el margen teórico para todo el catálogo usando los precios y costos existentes
margen_calculado = (df_productos['precio_lista'] - df_productos['costo_unitario']) / df_productos['costo_unitario']

# 2. Inyectamos este cálculo ÚNICAMENTE en las celdas donde margen_ganancia es nulo (NaN)
df_productos['margen_ganancia'] = df_productos['margen_ganancia'].fillna(margen_calculado)

# 3. Redondeamos a 4 decimales para mantener la coherencia del formato (ej: 0.2460)
df_productos['margen_ganancia'] = df_productos['margen_ganancia'].round(4)

# 4. Verificamos el resultado específico del Mouse Inalámbrico y otros posibles nulos
print("--- Márgenes Restaurados ---")
display(df_productos[['nombre_producto', 'costo_unitario', 'precio_lista', 'margen_ganancia']].head(15))

--- Márgenes Restaurados ---


,nombre_producto,costo_unitario,precio_lista,margen_ganancia
0,Televisor 42'',2676.44,3848.45,0.4379
1,Auriculares BT,2420.13,3052.45,0.2915
2,Zapatillas Running,1360.93,2778.60,0.2460
3,Remera Deportiva,1821.59,3570.69,-0.0500
4,Licuadora 600W,7567.49,9217.83,0.3358
5,Plancha de Pelo,1895.89,3626.34,0.2270
6,Galletitas Integrales,3219.93,3907.39,0.2135
7,Yerba 1kg,4964.48,8397.23,0.1320
8,Short Entrenamiento,3400.90,4397.36,0.2930
9,Campera Impermeable,7546.56,322992.77,41.8000


In [33]:
# 1. Identificamos qué filas tienen un margen ilógico (mayor al 100%, es decir, > 1)
mascara_outliers = df_productos['margen_ganancia'] > 1

# 2. Corregimos el margen dividiéndolo por 100
df_productos.loc[mascara_outliers, 'margen_ganancia'] = df_productos.loc[mascara_outliers, 'margen_ganancia'] / 100

# 3. Recalculamos el precio de lista ÚNICAMENTE para los productos que tenían este error
df_productos.loc[mascara_outliers, 'precio_lista'] = df_productos.loc[mascara_outliers, 'costo_unitario'] * (1 + df_productos.loc[mascara_outliers, 'margen_ganancia'])

# 4. Volvemos a redondear el precio por prolijidad
df_productos['precio_lista'] = df_productos['precio_lista'].round(2)

# 5. Verificamos cómo quedó la Campera Impermeable ahora
print("--- Corrección de Outlier ---")
display(df_productos[df_productos['nombre_producto'] == 'Campera Impermeable'][['nombre_producto', 'costo_unitario', 'margen_ganancia', 'precio_lista']])

--- Corrección de Outlier ---


,nombre_producto,costo_unitario,margen_ganancia,precio_lista
9,Campera Impermeable,7546.56,0.418,10701.02


In [34]:
df_productos

,id_producto,nombre_producto,marca,categoria,costo_unitario,precio_lista,stock_actual,activo,margen_ganancia,proveedor,cod_interno,ubicacion_deposito,unidad_medida,fecha_ingreso,peso_kg
0,101,Televisor 42'',Sony,Electrónica,2676.44,3848.45,87,False,0.437900,Proveedor A,INT-7407,Deposito 1,Unidad,2024-02-26,3.95
1,102,Auriculares BT,Sony,Electrónica,2420.13,3052.45,34,False,0.291500,Proveedor B,INT-7221,Deposito 2,Unidad,2023-12-05,8.42
2,103,Zapatillas Running,Nike,Deportes,1360.93,2778.60,75,False,0.246000,Proveedor A,INT-6337,Deposito 1,Unidad,2024-07-01,13.33
3,104,Remera Deportiva,Adidas,Indumentaria,1821.59,3570.69,48,False,-0.050000,Proveedor A,INT-5761,Externo,Unidad,2023-08-16,23.69
4,105,Licuadora 600W,Philips,Hogar,7567.49,9217.83,77,False,0.335800,Proveedor B,INT-6786,Deposito 1,Unidad,2023-08-24,6.22
5,106,Plancha de Pelo,Philips,Hogar,1895.89,3626.34,140,False,0.227000,Proveedor A,INT-6370,Externo,Unidad,2023-12-30,17.73
6,107,Galletitas Integrales,Molinos,Alimentos,3219.93,3907.39,158,False,0.213500,Proveedor C,INT-9767,Deposito 1,Unidad,2023-01-18,15.51
7,108,Yerba 1kg,Molinos,Alimentos,4964.48,8397.23,129,False,0.132000,Proveedor A,INT-6091,Deposito 2,Unidad,2023-08-24,11.00
8,109,Short Entrenamiento,Reebok,Indumentaria,3400.90,4397.36,143,True,0.293000,Proveedor A,INT-1146,Deposito 1,Unidad,2022-01-24,4.45
9,110,Campera Impermeable,Topper,Indumentaria,7546.56,10701.02,119,False,0.418000,Proveedor A,INT-5696,Deposito 1,Unidad,2024-02-14,24.72


In [35]:
# Exportar el DataFrame de Productos limpios
# Es importante el encoding='utf-8' para que caracteres como la 'ñ' o acentos no se rompan
df_productos.to_csv('Processed/productos_procesado.csv', index=False, encoding='utf-8')

print("✅ df_productos exportado exitosamente a Processed/productos_procesado.csv")

✅ df_productos exportado exitosamente a Processed/productos_procesado.csv
